In [8]:
spark.stop()

In [1]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("User Defined Functions")
    .master("local[*]")
    .getOrCreate()
)

spark


In [3]:
# Read employee data

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

emp = spark.read.format("csv").option("header", True).schema(emp_schema).load("scratch/input/emp.csv")

emp.rdd.getNumPartitions()

1

In [4]:
# Create a function to generate 10% of Salary as Bonus

def bonus(salary):
    return int(salary) * 0.1

In [ ]:
# pyspark.sql.functions.udf is used to convert a regular Python function into a UDF that can be used in Spark DataFrame operations. 
# The udf function takes the Python function as an argument and returns a new function that can be applied to columns in a DataFrame.
# v1
from pyspark.sql.functions import udf

bonus_udf = udf(bonus)
emp.withColumn("bonus", bonus_udf(emp.salary)).show()


In [7]:
# # Register as UDF for Spark SQL
# v2
from pyspark.sql.functions import udf

spark.udf.register("bonus_sql_udf", bonus, "double")

<function __main__.bonus(salary)>

In [ ]:
# Create new column as bonus using UDF
from pyspark.sql.functions import expr

emp.withColumn("bonus", expr("bonus_sql_udf(salary)")).show()
# emp.withColumn("bonus2", bonus_udf("salary")).show()
# emp.drop("bonus2")

DataFrame[employee_id: string, department_id: string, name: string, age: string, gender: string, salary: string, hire_date: string]

In [ ]:
# Create new column as bonus without UDF
# v3
emp.withColumn("bonus", expr("salary * 0.1")).show()

In [26]:
# Stop Spark Session

spark.stop()